In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../')
sys.path.append('../../')
import numpy as np
import torch
from MINAR.ComputationGraph import ComputationGraph, Circuit
from model.CustomLosses import MultiplicativeLoss
import torch_geometric as pyg
import networkx as nx
from model.MinAggGNN import MinAggGNN
import matplotlib.pyplot as plt

device = torch.device('cuda')
L = 2
m = 2
epochs = 20000
eta = 0.001
seed = 0

model = MinAggGNN(1, 8, L, 1, edge_dim = 1)
state_dict = torch.load(f'../model_progress/bellman_ford/seed_{seed}/model_final.pt')
model.load_state_dict(state_dict)
model.eval()
model.to(device)

c:\Users\heje197\AppData\Local\miniconda3\envs\minar\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MinAggGNN(1, 1, num_layers=2)

In [2]:
test_data = torch.load('../data/test_data.pt', map_location=device, weights_only=False)
test_loader = pyg.loader.DataLoader(test_data, batch_size=len(test_data), shuffle=False)
corrupted_data = torch.load('../data/test_data.pt', map_location=device, weights_only=False)
for data_corr in corrupted_data:
    data_corr.x = torch.zeros_like(data_corr.x, device=device)
    data_corr.x[0] = 0.
    data_corr.edge_attr = torch.zeros_like(data_corr.edge_attr, device=device)
criterion = MultiplicativeLoss()
mse_criterion = torch.nn.MSELoss()
num_reachable_test_nodes = sum([data.reachable.sum() for data in test_loader])

In [7]:
def find_sufficient_K(G, model, test_loader, which='EAP-IG', max_K=100, eps=1e-3, verbose=True):
    model_loss = 0.0
    for data in test_loader:
        out = model(data.x, data.edge_index, edge_attr = data.edge_attr)
        model_loss += float(criterion(out[data.reachable].flatten(), data.y[data.reachable]).detach()) / num_reachable_test_nodes
    K = 1
    best_K = max_K
    best_edges = max_K
    circuit_loss = np.inf
    while K <= max_K and circuit_loss > model_loss + eps:
        C_tmp = Circuit(model, G, K=K, key=which)
        total_loss = 0.0
        for data in test_loader:
            out = C_tmp.forward(data)
            total_loss += float(criterion(out[data.reachable].flatten(), data.y[data.reachable]).detach()) / num_reachable_test_nodes
        if total_loss < circuit_loss:
            circuit_loss = total_loss
            best_K = K
            best_edges = C_tmp.number_of_edges()
        if circuit_loss < model_loss + eps:
            break
        K += 1
    return best_K, best_edges, circuit_loss

In [8]:
seeds = [0,1,2,3,4]
check_epochs = [1000, 2000, 3000]
which = ['weight_grad', 'EAP-IG']
for seed in seeds:
    checkpoints = torch.load(f'../model_progress/bellman_ford/seed_{seed}/model_checkpoints.pt')
    for idx, check_epoch in enumerate(check_epochs):
        checkpoint_idx = check_epoch // 100 - 1
        model.load_state_dict(checkpoints[checkpoint_idx])
        G_mid = ComputationGraph(model)
        G_mid.add_inputs({'edge_attr' : [1, model.convs[0].agg_mlp.lins[0].weight[:,-1]],
              'input_self' : [3, model.convs[0].up_mlp.lins[0].weight[:,-1]]})
        G_mid.add_residual_connections({'edge_attr' : [5, model.convs[1].agg_mlp.lins[0].weight[:,-1].reshape(1,-1).cpu().detach()]})
        G_mid.add_residual_connections({4 : [7, model.convs[1].up_mlp.lins[0].weight[:,-8:].T.cpu().detach()]})
        G_mid.calculate_scores(test_data, corrupted_data, mse_criterion, which = 'weight_grad')
        G_mid.calculate_scores(test_data, corrupted_data, mse_criterion, which = 'EAP-IG', steps=20)
        for which in ['weight_grad', 'EAP-IG']:
            K, best_edges, loss = find_sufficient_K(G_mid, model, test_loader, which=which, verbose=False)
            print(f'Seed: {seed}, Score Method: {which}, Epoch: {check_epoch}, Best K: {K}, Best Edges: {best_edges}, Circuit Loss: {loss:.4f}')

Seed: 0, Score Method: weight_grad, Epoch: 1000, Best K: 92, Best Edges: 91, Circuit Loss: 0.0585
Seed: 0, Score Method: EAP-IG, Epoch: 1000, Best K: 72, Best Edges: 71, Circuit Loss: 0.0615
Seed: 0, Score Method: weight_grad, Epoch: 2000, Best K: 34, Best Edges: 34, Circuit Loss: 0.0561
Seed: 0, Score Method: EAP-IG, Epoch: 2000, Best K: 31, Best Edges: 29, Circuit Loss: 0.0565
Seed: 0, Score Method: weight_grad, Epoch: 3000, Best K: 12, Best Edges: 12, Circuit Loss: 0.0577
Seed: 0, Score Method: EAP-IG, Epoch: 3000, Best K: 10, Best Edges: 10, Circuit Loss: 0.0543
Seed: 1, Score Method: weight_grad, Epoch: 1000, Best K: 76, Best Edges: 76, Circuit Loss: 0.0654
Seed: 1, Score Method: EAP-IG, Epoch: 1000, Best K: 60, Best Edges: 58, Circuit Loss: 0.0634
Seed: 1, Score Method: weight_grad, Epoch: 2000, Best K: 38, Best Edges: 38, Circuit Loss: 0.0588
Seed: 1, Score Method: EAP-IG, Epoch: 2000, Best K: 31, Best Edges: 31, Circuit Loss: 0.0578
Seed: 1, Score Method: weight_grad, Epoch: 30